# 🚀 pgVectorDB v0.0.6 — Unified API Quick Start

This notebook demonstrates the new **Unified Query API** introduced in pgVectorDB v0.0.6.

## What's New

- **Single entry point**: `db.query("...")` for all search methods
- **Fluent API**: Chain methods for clean, readable code
- **Search modes**: Semantic, Keyword (BM25/FTS), Hybrid, Trigram
- **Multimodal support**: Search across text, numbers, categories
- **Query analysis**: Built-in explain and analyze capabilities

### Prerequisites
- PostgreSQL with `pgvector` extension
- Python dependencies: `pip install pgvectordb[huggingface]`

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from langchain_core.documents import Document
from pgvectordb import pgVectorDB, Config, SearchMethod

print("✅ Imports successful")

## 1. Initialize Database

Connect to PostgreSQL and create the vector store.

In [ ]:
rag = pgVectorDB(
    collection_name="nb_unified_api",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag.initialize(overwrite_existing=True)
print("✅ Database initialized")

## 2. Add Sample Documents

We'll add documents with rich metadata for filtering and multimodal search.

In [ ]:
docs = [
    Document(
        page_content="PostgreSQL is a powerful open-source relational database system with advanced features.",
        metadata={"category": "database", "year": 2024, "priority": 9, "price": 0},
    ),
    Document(
        page_content="pgvector adds vector similarity search capabilities to PostgreSQL for AI applications.",
        metadata={"category": "database", "year": 2024, "priority": 10, "price": 0},
    ),
    Document(
        page_content="Machine learning models convert text into dense vector embeddings for semantic search.",
        metadata={"category": "ai", "year": 2023, "priority": 8, "price": 50000},
    ),
    Document(
        page_content="RAG combines retrieval and generation for accurate AI responses with sources.",
        metadata={"category": "ai", "year": 2024, "priority": 9, "price": 75000},
    ),
    Document(
        page_content="Docker containers package applications with all dependencies for consistent deployment.",
        metadata={"category": "devops", "year": 2022, "priority": 7, "price": 299},
    ),
    Document(
        page_content="Kubernetes orchestrates containerized workloads at scale across clusters.",
        metadata={"category": "devops", "year": 2023, "priority": 8, "price": 499},
    ),
]

ids = await rag.add_documents(docs)
print(f"✅ Added {len(ids)} documents")

## 3. Semantic Search (Default)

The default search mode uses vector similarity.

In [ ]:
# Simple semantic search
results = await rag.query("vector database AI").limit(3).to_list()

print("🔍 Semantic Search Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

## 4. Keyword Search with BM25

Use `.search_mode(SearchMethod.KEYWORD)` for BM25 ranking.

In [ ]:
from pgvectordb import SearchMethod

results = await (
    rag.query("database search")
    .search_mode(SearchMethod.KEYWORD)
    .bm25_params(k1=1.2, b=0.75)  # BM25 parameters
    .limit(3)
    .to_list()
)

print("🔤 BM25 Keyword Search:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

## 5. Hybrid Search (Vector + Keyword)

Combine semantic and keyword signals with weighted fusion or RRF.

In [ ]:
# Weighted fusion
results = await (
    rag.query("vector database performance")
    .search_mode(SearchMethod.HYBRID)
    .weights(semantic=0.7, keyword=0.3)  # 70% semantic, 30% keyword
    .limit(3)
    .to_list()
)

print("⚡ Hybrid Search (Weighted):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

In [ ]:
# RRF fusion (Reciprocal Rank Fusion)
results = await (
    rag.query("vector database performance")
    .search_mode(SearchMethod.HYBRID)
    .rrf(k=60)  # RRF constant
    .limit(3)
    .to_list()
)

print("⚡ Hybrid Search (RRF):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

## 6. Metadata Filtering

Use `.where()` for MongoDB-style filtering.

In [ ]:
# Filter by category and year
results = await (
    rag.query("AI systems")
    .where({"category": "ai", "year": {"$gte": 2023}})
    .limit(5)
    .to_list()
)

print("📋 Filtered Results (category=ai, year≥2023):")
for r in results:
    print(f"  • {r['content'][:50]}... ({r['metadata']})")

In [ ]:
# Complex filter with $and/$or
results = await (
    rag.query("database")
    .where({
        "$or": [
            {"category": "database"},
            {"priority": {"$gte": 9}}
        ]
    })
    .limit(5)
    .to_list()
)

print("📋 Complex Filter Results:")
for r in results:
    print(f"  • {r['content'][:50]}... (priority: {r['metadata'].get('priority')})")

## 7. Query Parameter Tuning

Fine-tune vector search parameters.

In [ ]:
# HNSW parameter tuning
results = await (
    rag.query("vector embeddings")
    .ef(100)  # Increase candidate pool for better recall
    .refine_factor(2)  # Oversample and rerank
    .limit(3)
    .to_list()
)

print("🎛️ Tuned Search Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:50]}...")

## 8. Query Analysis

Analyze query execution with `.explain_plan()` and `.analyze_plan()`.

In [ ]:
# Explain plan (no execution)
plan = rag.query("test").where({"category": "ai"}).explain_plan()

print("📊 Query Plan:")
for key, value in plan.items():
    print(f"  {key}: {value}")

In [ ]:
# Analyze with execution metrics
metrics = await (
    rag.query("vector database")
    .where({"category": "database"})
    .limit(5)
    .analyze_plan()
)

print("📈 Execution Metrics:")
for key, value in metrics.items():
    if key != "config":  # Skip verbose config
        print(f"  {key}: {value}")

## 9. Output Formats

Get results in different formats.

In [ ]:
# As pandas DataFrame
df = await rag.query("test").limit(3).to_pandas()
print("📊 Pandas DataFrame:")
print(df[['content', 'score']].head())

## 10. Comparison: Old vs New API

The legacy API still works for backward compatibility.

In [ ]:
# Old API (still works)
old_results = await rag.semantic_search("database", k=3)
print("Old API:", len(old_results), "results")

# New Unified API
new_results = await rag.query("database").limit(3).to_list()
print("New API:", len(new_results), "results")

## Summary

The Unified API provides:

- **Single entry point**: `db.query("...")`
- **Search modes**: `.semantic()`, `.keyword()`, `.hybrid()`, `.trigram()`
- **Configuration**: `.where()`, `.limit()`, `.ef()`, `.bm25_params()`, `.rrf()`
- **Output formats**: `.to_list()`, `.to_pandas()`, `.to_arrow()`
- **Analysis**: `.explain_plan()`, `.analyze_plan()`

For more details, see the [Migration Guide](../docs/user_guide/migration_guide.md).

In [ ]:
# Cleanup
await rag.delete_table()
await rag.close()
print("🧹 Cleaned up")